# Querying Files with Spark SQL

In a lakehouse, raw data often lands on cloud storage before it is cleaned, modeled, or registered as a table. Spark SQL lets you query files **directly** — no table registration needed. This is the foundation of the Bronze layer: ingest first, register later.

In this notebook we'll cover:
- The backtick syntax for querying files directly in SQL
- Which formats support direct querying and how they behave differently
- Reading directories and glob patterns for multi-file datasets
- Creating temporary and permanent views over files
- When to use direct queries vs views vs tables

## Direct File Querying — The Backtick Syntax

Spark SQL supports a special syntax for querying files without registering them as tables:

```sql
SELECT * FROM format.`path/to/file_or_directory`
```

- `format` — the file format keyword: `delta`, `parquet`, `json`, `csv`, `text`, `binaryFile`
- `path` — an absolute path, wrapped in **backticks** (not quotes)

**Examples:**

```sql
-- Read a single Parquet file
SELECT * FROM parquet.`dbfs:/mnt/raw/orders/2024-01-15.parquet`;

-- Read all files in a Delta table directory
SELECT * FROM delta.`dbfs:/mnt/bronze/events/`;

-- Read all JSON files in a directory
SELECT * FROM json.`dbfs:/mnt/raw/users/`;

-- Read a CSV file
SELECT * FROM csv.`dbfs:/mnt/raw/products.csv`;
```

> **Exam tip:** The backtick is essential — without it, Spark interprets the path as a table name and throws an error. Single quotes won't work here; it must be backticks.

## Format Behaviour — What to Expect

Different formats behave very differently when queried directly:

| Format | Schema | Notes |
|--------|--------|-------|
| **`delta`** | Self-describing — exact schema from transaction log | Best option for reliable direct reads |
| **`parquet`** | Self-describing — schema embedded in file footer | Reliable, but no ACID; concurrent writes can produce inconsistent reads |
| **`json`** | Inferred by sampling records | All fields become strings if types are inconsistent; multiline JSON needs a reader option |
| **`csv`** | No schema — everything is a string by default | `header` row used as column names if present; needs explicit schema for types |
| **`text`** | Single column called `value` (StringType) | Entire line per row — useful for logs, unstructured data |
| **`binaryFile`** | Fixed schema: `path`, `modificationTime`, `length`, `content` (BinaryType) | For images, PDFs, audio; content is raw bytes |

### Self-Describing vs Schema-on-Read

**Self-describing formats** (Delta, Parquet) embed the schema in the file — Spark reads it directly, no guessing required.

**Schema-on-read formats** (JSON, CSV) require Spark to infer the schema by sampling data. This is unreliable at scale:
- Sample may not include all fields
- Type inference can be wrong (e.g., `"123"` inferred as Integer, then a later record has `"N/A"`)
- Inference adds overhead

For production pipelines, always define an explicit schema when reading CSV or JSON.

## Reading Directories and Glob Patterns

When a path points to a **directory**, Spark reads **all files** in that directory that match the format — it does not recurse into subdirectories by default.

```sql
-- Reads every Parquet file in the directory (non-recursive)
SELECT * FROM parquet.`dbfs:/mnt/raw/orders/`;
```

**Glob patterns** let you match a subset of files:

```sql
-- Only January 2024 files
SELECT * FROM parquet.`dbfs:/mnt/raw/orders/2024-01-*`;

-- All Q1 2024 files across month subdirectories
SELECT * FROM parquet.`dbfs:/mnt/raw/orders/2024-0{1,2,3}/*`;

-- All files two levels deep
SELECT * FROM json.`dbfs:/mnt/raw/events/*/*`;
```

### The `_metadata` Column

When reading a directory with multiple files, Spark automatically makes a hidden `_metadata` column available. It lets you see which file each row came from:

```sql
SELECT _metadata.file_path,
       _metadata.file_name,
       _metadata.file_modification_time,
       *
FROM   parquet.`dbfs:/mnt/raw/orders/`;
```

This is useful for auditing, debugging, and building incremental pipelines that track which source files have been processed.

## Creating Views Over Files

Querying files with the backtick syntax is fine for one-off exploration. For repeated use, create a **view** — it gives the file query a name that behaves like a table, without moving any data.

### Temporary Views

Exist only for the current SparkSession. Dropped automatically when the session ends.

```sql
CREATE OR REPLACE TEMP VIEW raw_orders
AS SELECT * FROM parquet.`dbfs:/mnt/raw/orders/`;

-- Now query it like a table
SELECT region, COUNT(*) FROM raw_orders GROUP BY region;
```

### Global Temporary Views

Shared across sessions within the same Spark application. Accessible via `global_temp.view_name`.

```sql
CREATE OR REPLACE GLOBAL TEMP VIEW raw_orders
AS SELECT * FROM parquet.`dbfs:/mnt/raw/orders/`;

SELECT * FROM global_temp.raw_orders;
```

### Persistent Views

Registered in the metastore — survive session restarts, visible to all users with access to the schema.

```sql
CREATE OR REPLACE VIEW bronze.raw_orders
AS SELECT * FROM parquet.`dbfs:/mnt/raw/orders/`;
```

### View vs Table

| | View | Table |
|---|---|---|
| Data stored | No — query re-executes each time | Yes — data is physically stored |
| Performance | Slower (re-reads source on every query) | Faster (data already written and optimized) |
| Always fresh | Yes — reflects current files | No — needs refresh/overwrite to pick up new data |
| Use case | Alias for a file location; exploration | Persistent, optimized, production-ready data |

> **Exam tip:** `DESCRIBE EXTENDED view_name` shows `Type: VIEW` and the underlying query text. `DESCRIBE EXTENDED table_name` shows `Type: MANAGED` or `Type: EXTERNAL`.

## Reading with Explicit Options (PySpark)

The backtick SQL syntax uses default options. For CSV and JSON, you almost always need to override them. Use the PySpark `DataFrameReader` with `.option()` calls:

```python
# CSV with explicit schema and options
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, DateType

schema = StructType([
    StructField("order_id",   StringType(), True),
    StructField("customer_id",StringType(), True),
    StructField("amount",     DoubleType(), True),
    StructField("order_date", DateType(),   True),
])

df = (spark.read
           .format("csv")
           .schema(schema)
           .option("header", True)
           .option("sep", ",")
           .option("dateFormat", "yyyy-MM-dd")
           .option("nullValue", "N/A")
           .load("dbfs:/mnt/raw/orders.csv"))

# JSON — multiline documents (one JSON object spans multiple lines)
df_json = (spark.read
                .format("json")
                .option("multiline", True)
                .load("dbfs:/mnt/raw/events/"))
```

**Common CSV options:**

| Option | Default | Purpose |
|--------|---------|--------|
| `header` | `false` | Use first row as column names |
| `sep` | `,` | Field delimiter |
| `nullValue` | `""` | String to treat as null |
| `dateFormat` | `yyyy-MM-dd` | Date parsing pattern |
| `inferSchema` | `false` | Let Spark guess types (slow, unreliable) |
| `mode` | `PERMISSIVE` | How to handle malformed records: `PERMISSIVE`, `DROPMALFORMED`, `FAILFAST` |

## Hands-On: Direct Queries and Views

In [ ]:
# Setup — write sample files to DBFS so we can query them
import json

# Write some JSON records
records = [
    {"order_id": "ORD001", "customer_id": "CUST01", "amount": 120.50, "region": "UK"},
    {"order_id": "ORD002", "customer_id": "CUST02", "amount": 340.00, "region": "US"},
    {"order_id": "ORD003", "customer_id": "CUST01", "amount":  89.99, "region": "UK"},
]

dbutils.fs.mkdirs("dbfs:/tmp/demo/orders_json/")
dbutils.fs.put(
    "dbfs:/tmp/demo/orders_json/orders.json",
    "\n".join(json.dumps(r) for r in records),
    overwrite=True
)
print("Sample JSON written to dbfs:/tmp/demo/orders_json/")

In [ ]:
# Write sample Parquet and Delta files
from pyspark.sql import Row

df = spark.createDataFrame([
    Row(order_id="ORD001", customer_id="CUST01", amount=120.50, region="UK"),
    Row(order_id="ORD002", customer_id="CUST02", amount=340.00, region="US"),
    Row(order_id="ORD003", customer_id="CUST01", amount= 89.99, region="UK"),
])

df.write.mode("overwrite").parquet("dbfs:/tmp/demo/orders_parquet/")
df.write.format("delta").mode("overwrite").save("dbfs:/tmp/demo/orders_delta/")
print("Parquet and Delta files written")

In [ ]:
# Direct file query — JSON
spark.sql("SELECT * FROM json.`dbfs:/tmp/demo/orders_json/`").show()

In [ ]:
# Direct file query — Parquet with _metadata column
spark.sql("""
  SELECT _metadata.file_name,
         _metadata.file_modification_time,
         order_id,
         amount
  FROM   parquet.`dbfs:/tmp/demo/orders_parquet/`
""").show(truncate=False)

In [ ]:
# Direct file query — Delta
spark.sql("SELECT * FROM delta.`dbfs:/tmp/demo/orders_delta/`").show()

In [ ]:
# Create a temp view over the JSON files
spark.sql("""
  CREATE OR REPLACE TEMP VIEW raw_orders_vw
  AS SELECT * FROM json.`dbfs:/tmp/demo/orders_json/`
""")

# Query the view — looks like a table
spark.sql("""
  SELECT region, COUNT(*) AS cnt, ROUND(SUM(amount), 2) AS total
  FROM   raw_orders_vw
  GROUP BY region
""").show()

In [ ]:
# DESCRIBE EXTENDED shows it's a VIEW, not a table
spark.sql("DESCRIBE EXTENDED raw_orders_vw").show(truncate=False)

In [ ]:
# PySpark read with explicit schema (CSV pattern)
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

schema = StructType([
    StructField("order_id",    StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("amount",      DoubleType(), True),
    StructField("region",      StringType(), True),
])

df_typed = (spark.read
                 .format("json")
                 .schema(schema)
                 .load("dbfs:/tmp/demo/orders_json/"))

df_typed.printSchema()
df_typed.show()

In [ ]:
# Cleanup
dbutils.fs.rm("dbfs:/tmp/demo/", recurse=True)
spark.sql("DROP VIEW IF EXISTS raw_orders_vw")

## When to Use Direct Queries vs Views vs Tables

```
Goal                                      → Approach
──────────────────────────────────────────────────────
One-off exploration of raw files          → Direct backtick query
Reusable alias for a file path            → Temp view (session-scoped)
Shared alias across users/sessions        → Persistent view (registered in metastore)
Repeated production reads, needs speed   → Register as Delta table + OPTIMIZE
```

In a medallion pipeline:
- **Bronze** often starts as direct file queries or Auto Loader reads (next notebook)
- **Silver** is always a registered Delta table — cleaned, typed, constrained
- **Gold** is always a registered Delta table — aggregated, indexed, ready for BI

## Summary

- The backtick syntax `SELECT * FROM format.\`path\`` lets you query files directly without registering a table. The backtick is mandatory — not single quotes.
- Self-describing formats (Delta, Parquet) give reliable schemas. Schema-on-read formats (JSON, CSV) require explicit schemas in production.
- Pointing a path to a **directory** reads all files in it. **Glob patterns** let you filter which files to include.
- The `_metadata` column reveals the source file path and modification time for each row — useful for auditing and incremental tracking.
- **Temp views** alias a file query for the session. **Persistent views** register in the metastore for all users. Neither moves data.
- `DESCRIBE EXTENDED view_name` shows `Type: VIEW` and the underlying query — a quick way to distinguish views from tables on the exam.
- In production, always define an explicit schema when reading CSV or JSON — never rely on `inferSchema`.

Next: `05-writing-to-delta-tables.ipynb` — INSERT, COPY INTO, and overwrite strategies for loading data into Delta.